In [4]:
from pathlib import Path
import pandas as pd 

files = list(Path("/opt/airflow/data/source").glob("*"))
data = pd.read_csv(
    "/opt/airflow/data/source/household_power_consumption.txt",
    sep=";"
)



/tmp/ipykernel_229/4066741963.py:5: DtypeWarning: Columns (2,3,4,5,6,7) have mixed types. Specify dtype option on import or set low_memory=False.
  data = pd.read_csv(


In [5]:
from pathlib import Path

BATCH_SIZE = 1000

output_dir = Path("/opt/airflow/data/raw_power_batches")
output_dir.mkdir(parents=True, exist_ok=True)

for start in range(0, len(data), BATCH_SIZE):
    end = start + BATCH_SIZE
    batch = data.iloc[start:end]

    batch_number = start // BATCH_SIZE + 1
    batch_file = output_dir / f"batch_{batch_number:05d}.csv"

    batch.to_csv(batch_file, index=False)



In [6]:
files = list(Path("/opt/airflow/data/raw_power_batches").glob("*.csv"))

print("number of batches:", len(files))
print(files[:5])

number of batches: 2076
[PosixPath('/opt/airflow/data/raw_power_batches/batch_00735.csv'), PosixPath('/opt/airflow/data/raw_power_batches/batch_01923.csv'), PosixPath('/opt/airflow/data/raw_power_batches/batch_00464.csv'), PosixPath('/opt/airflow/data/raw_power_batches/batch_01922.csv'), PosixPath('/opt/airflow/data/raw_power_batches/batch_01961.csv')]


In [7]:
import os
os.environ["HADOOP_USER_NAME"] = "root"

from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("PowerConsumptionETL") \
    .master("local[*]") \
    .config("spark.hadoop.fs.defaultFS", "hdfs://hadoop-namenode:9000") \
    .getOrCreate()


In [8]:
df = spark.read.csv(
    "file:///opt/airflow/data/raw_power_batches",
    header=True,
    inferSchema=True
)

df.show(5)
print(df.count())

+----------+-------------------+-------------------+---------------------+-------+----------------+--------------+--------------+--------------+
|      Date|               Time|Global_active_power|Global_reactive_power|Voltage|Global_intensity|Sub_metering_1|Sub_metering_2|Sub_metering_3|
+----------+-------------------+-------------------+---------------------+-------+----------------+--------------+--------------+--------------+
|23/11/2008|2026-04-28 08:44:00|              0.406|                0.066|244.180|           1.800|         0.000|         1.000|           0.0|
|23/11/2008|2026-04-28 08:45:00|              0.404|                0.068|244.720|           1.800|         0.000|         1.000|           0.0|
|23/11/2008|2026-04-28 08:46:00|              0.858|                0.238|244.570|           5.000|         0.000|         2.000|           0.0|
|23/11/2008|2026-04-28 08:47:00|              0.444|                0.052|244.900|           2.000|         0.000|         1.000| 

In [ ]:
hdfs_bronze_path = "hdfs://hadoop-namenode:9000/user/root/datalake/bronze/power_consumption"

df.write \
    .mode("overwrite") \
    .parquet(hdfs_bronze_path)
